In [0]:
WITH params AS (
  SELECT 30 AS p_window_days
),
clusters_latest AS (
  SELECT
    workspace_id,
    cluster_id,
    cluster_name,
    cluster_source,
    change_time,
    delete_time,
    ROW_NUMBER() OVER (
      PARTITION BY workspace_id, cluster_id
      ORDER BY change_time DESC
    ) AS rn
  FROM system.compute.clusters
),
clusters_current AS (
  SELECT
    workspace_id,
    cluster_id,
    cluster_name,
    cluster_source,
    delete_time
  FROM clusters_latest
  WHERE rn = 1
),
cluster_counts AS (
  SELECT
    COUNT(*) AS clusters_known,
    SUM(CASE WHEN delete_time IS NULL THEN 1 ELSE 0 END) AS clusters_not_deleted,
    SUM(
      CASE
        WHEN delete_time IS NULL AND upper(COALESCE(cluster_source, '')) = 'JOB'
        THEN 1
        ELSE 0
      END
    ) AS job_clusters_not_deleted
  FROM clusters_current
),
node_30d AS (
  SELECT
    COUNT(DISTINCT cluster_id) AS active_clusters_30d,
    COUNT(DISTINCT instance_id) AS active_instances_30d,
    ROUND(AVG(cpu_user_percent), 2) AS avg_cpu_user_percent_30d,
    ROUND(AVG(mem_used_percent), 2) AS avg_mem_used_percent_30d,
    ROUND(SUM(network_sent_bytes) / POW(1024, 3), 3) AS network_sent_gb_30d,
    ROUND(SUM(network_received_bytes) / POW(1024, 3), 3) AS network_received_gb_30d,
    MAX(end_time) AS last_observed_utc
  FROM system.compute.node_timeline n
  CROSS JOIN params p
  WHERE n.start_time >= timestampadd(DAY, -p.p_window_days, current_timestamp())
),
node_last_hour AS (
  SELECT
    COUNT(DISTINCT cluster_id) AS active_clusters_last_hour,
    COUNT(DISTINCT instance_id) AS active_instances_last_hour
  FROM system.compute.node_timeline
  WHERE end_time >= timestampadd(HOUR, -1, current_timestamp())
)
SELECT
  c.clusters_known,
  c.clusters_not_deleted,
  c.job_clusters_not_deleted,
  n.active_clusters_30d,
  n.active_instances_30d,
  h.active_clusters_last_hour,
  h.active_instances_last_hour,
  n.avg_cpu_user_percent_30d,
  n.avg_mem_used_percent_30d,
  n.network_sent_gb_30d,
  n.network_received_gb_30d,
  n.last_observed_utc,
  from_utc_timestamp(n.last_observed_utc, 'Asia/Seoul') AS last_observed_kst,
  CASE
    WHEN n.last_observed_utc IS NULL THEN 'NO_DATA'
    WHEN n.last_observed_utc < timestampadd(HOUR, -2, current_timestamp()) THEN 'STALE'
    ELSE 'HEALTHY'
  END AS telemetry_freshness,
  current_timestamp() AS generated_at_utc,
  from_utc_timestamp(current_timestamp(), 'Asia/Seoul') AS generated_at_kst
FROM cluster_counts c
CROSS JOIN node_30d n
CROSS JOIN node_last_hour h;
